# Notebook 2: Data retrieval from publications

**LLMs in Bioinformatics workshop** | ~90 minutes | internet needed | any AI assistant (Claude, ChatGPT, Gemini, Colab's built-in assistant)

Most of the data biologists need is not in a tidy database. It sits inside **papers**: in the text, in figure legends and in supplementary Excel files. Getting it out by hand is slow and error-prone. An AI assistant can write the code for you (**vibe coding**), but AI code usually *runs*, so wrong data looks right.

In this notebook you retrieve the data that the rest of the workshop uses, from a real paper, and you learn to **ask clearly** and **check before you trust**.

**The papers.** Two papers, because the text and the data live in different places:

- **The data:** Nusinow et al. 2020, *Quantitative Proteomics of the Cancer Cell Line Encyclopedia*, Cell. It measured the proteins of **375 cancer cell lines**, which the Cancer Cell Line Encyclopedia (**CCLE**) had already profiled for DNA (mutations, copy number) and RNA. Its tables are downloaded from the authors' lab site (Part 2). Its text is **not** served by Europe PMC (the first Colab run showed HTTP 500), which is itself a lesson about retrieval.
- **The text:** Jarnuczak et al. 2021, *An integrated landscape of protein expression in human cancer*, Scientific Data (PubMed Central **PMC8065022**, open access). It reanalysed public proteomics datasets of cancer cell lines and tumours. We read its text in Parts 1 and 3.

One cell line gives you all three layers:

| Layer | Measured as | Where you get it here |
| --- | --- | --- |
| Genomic (DNA) | mutations, copy number | a small class file prepared from DepMap (Part 4); the mutation coordinates are used in Notebook 3 |
| Transcriptomic (RNA) | RNA expression (log2 TPM) | the same class file |
| Proteomic | protein level (mass spectrometry) | **supplementary tables of the paper, hosted by the authors' lab (Part 2)** |

| Part | You retrieve | You check it by |
| --- | --- | --- |
| 1 | the **full text** of an open-access paper | its length, numbers you know, and a bad id |
| 2 | the **supplementary tables** (from the authors' lab site) as data frames | row and column counts, and one protein you know (ERBB2) |
| 3 | **facts from the text**, each with a quoted sentence | verifying that every quote really is in the paper |
| 4 | the **shared dataset file** for the next notebooks | uniqueness, and a result the paper itself reports |

### Time plan (90 min)

| Time | Part |
| --- | --- |
| 0:00-0:10 | **0. Setup and the prompt checklist** |
| 0:10-0:30 | **1. Get the paper** |
| 0:30-0:55 | **2. Supplementary tables** |
| 0:55-1:15 | **3. Facts with evidence** |
| 1:15-1:28 | **4. Build the shared dataset** |
| 1:28-1:30 | **5. Wrap-up** |

### The loop you repeat in every part

1. **Write your prompt** in a text cell. A checker gives feedback on what is missing.
2. **Paste it into your AI assistant** and copy the code it gives you into the marked cell.
3. **Run the check.** It tests the code against something you know. Do not skip it because the code ran.
4. **Hunt the bug** in a pre-made AI answer that looks fine but is wrong (available even without an assistant).
5. **Fix your prompt** so it would have prevented the bug.

No AI assistant available? Skip step 2 and use the safety-net functions that are provided in each part. You still learn the checking half.

## Part 0. Setup and the prompt checklist (10 min)

A prompt for code works best when it has these parts:

| Part | What to write | Example |
| --- | --- | --- |
| **Goal** | one sentence on the task | "Get the full text of an open-access paper." |
| **Inputs** | the exact source | "from the Europe PMC REST API, using the PMC id" |
| **Function** | the exact name, arguments and return value | "`def get_paper(pmcid) -> dict` with keys `title`, `sections`, `text`" |
| **Libraries** | which to use | "use `requests` and `xml.etree`" |
| **Edge cases** | what must happen when things go wrong | "raise a clear error for an id with no open full text" |
| **Checks** | ask for a self-test and for the assumptions made | "say what you assumed about the file layout" |

The **prompt checker** below is simple pattern matching. It looks for these parts; it does not understand your prompt. Treat it as a reminder list.

**For Part 4.** Your instructor will give you a small file called `ccle_panel.csv` (RNA, copy number and mutations for a panel of genes in the same cell lines). It is downloaded automatically from the course repository when the setup cell runs; if that fails, upload it to Colab (left sidebar, Files) before Part 4. Parts 1 to 3 do not need it.

In [ ]:
!pip -q install openpyxl requests pandas scipy pypdf 2>&1 | tail -1

# Class data files: downloaded automatically from the course site (nothing to upload).
import os, requests
CLASS_FILES_URL = "https://raw.githubusercontent.com/AmirAliLoghmani80/CBH-Workshop/main/data/"
def get_class_file(name):
    if os.path.exists(name):
        return name
    if "YOUR-USER" in CLASS_FILES_URL:
        print(f"NOTE: {name} not found and no download address is set. Upload it (left sidebar, Files) or ask the instructor.")
        return None
    r = requests.get(CLASS_FILES_URL + name, timeout=120)
    if r.status_code != 200 or r.content[:15].lstrip().lower().startswith((b"<!doctype", b"<html")):
        print(f"NOTE: could not download {name} (HTTP {r.status_code}). Upload it (left sidebar, Files) or ask the instructor.")
        return None
    with open(name, "wb") as f:
        f.write(r.content)
    print(f"downloaded {name} ({len(r.content)/1e6:.1f} MB)")
    return name

get_class_file("ccle_panel.csv")

In [ ]:
import io, os, re, json, time, zipfile, warnings
import numpy as np, pandas as pd
import requests
from scipy.stats import spearmanr
warnings.filterwarnings("ignore")

PMCID = "PMC8065022"        # Jarnuczak et al. 2021, Scientific Data (open access, CC BY): the paper whose text we read

def grade_prompt(prompt, required):
    """Look for the parts of a good prompt. `required` maps a part name to a regex."""
    found, missing = [], []
    for name, pattern in required.items():
        (found if re.search(pattern, prompt, flags=re.I) else missing).append(name)
    score = len(found) / len(required)
    print(f"Prompt checker: {len(found)}/{len(required)} parts found ({score:.0%})")
    print("  found  :", ", ".join(found) or "none")
    print("  missing:", ", ".join(missing) or "none")
    if len(prompt.split()) < 25:
        print("  note   : fewer than 25 words. Short prompts usually leave the AI guessing.")
    return score

COMMON = {
    "function signature": r"def \w+\s*\(|function (called|named)",
    "return value":       r"return|output|dict|dataframe|list of",
    "libraries":          r"requests|pandas|openpyxl|xml|zipfile|json",
    "edge cases":         r"error|raise|invalid|missing|nan|duplicate|exception|timeout|not found",
    "self-check":         r"assert|test|check|verify|assum|example",
}
print("helpers ready")

### Safety net (open this only if your own function will not work)

Reading these before you try defeats the exercise. They are here so that one broken part does not stop the next. Parts 2 to 4 use *your* function when it works and these when it does not.

In [ ]:
EPMC = "https://www.ebi.ac.uk/europepmc/webservices/rest"
PAPER_FILE = "paper.txt"        # optional: the paper's text saved by hand (last resort)

def _flat(el):
    return " ".join("".join(el.itertext()).split())

def _from_xml(content, pmcid):
    import xml.etree.ElementTree as ET
    root = ET.fromstring(content)
    title = _flat(root.find(".//article-title"))
    abstract = " ".join(_flat(a) for a in root.findall(".//abstract"))
    sections = []
    for sec in root.findall(".//body/sec"):
        t = sec.find("title")
        paras = [_flat(p) for p in sec.iter("p")]
        sections.append((_flat(t) if t is not None else "", " ".join(paras)))
    text = " ".join([abstract] + [body for _, body in sections])
    return {"pmcid": pmcid, "source": "Europe PMC XML", "title": title, "abstract": abstract, "sections": sections, "text": text}

def _from_text(text, pmcid, source):
    text = " ".join(text.split())
    return {"pmcid": pmcid, "source": source, "title": text[:200], "abstract": "", "sections": [("full text", text)], "text": text}

def get_paper(pmcid, timeout=90):
    pmcid = str(pmcid).strip().upper()
    if not re.fullmatch(r"PMC\d+", pmcid):
        raise ValueError(f"{pmcid!r} is not a PubMed Central id (expected something like PMC8065022)")
    problems = []
    # route 1: Europe PMC full-text XML (served only for some papers)
    try:
        r = requests.get(f"{EPMC}/{pmcid}/fullTextXML", timeout=timeout)
        if r.status_code == 200 and r.content.lstrip()[:1] == b"<":
            return _from_xml(r.content, pmcid)
        problems.append(f"full-text XML: HTTP {r.status_code}")
    except Exception as e:
        problems.append(f"full-text XML: {type(e).__name__}")
    # route 2: the PDF rendering of the paper, text taken out with pypdf
    try:
        r = requests.get(f"https://europepmc.org/api/getPdf?pmcid={pmcid}", timeout=timeout)
        if r.status_code == 200 and r.content[:4] == b"%PDF":
            from pypdf import PdfReader
            reader = PdfReader(io.BytesIO(r.content))
            return _from_text(" ".join((pg.extract_text() or "") for pg in reader.pages), pmcid, "Europe PMC PDF")
        problems.append(f"PDF: HTTP {r.status_code}")
    except Exception as e:
        problems.append(f"PDF: {type(e).__name__}")
    # route 3: a text file the person saved by hand
    if os.path.exists(PAPER_FILE):
        return _from_text(open(PAPER_FILE, encoding="utf-8", errors="ignore").read(), pmcid, f"file {PAPER_FILE}")
    raise RuntimeError(f"Could not get the text of {pmcid} ({'; '.join(problems)}). Last resort: save the paper's text "
                       f"(copy it from the PubMed Central page or from the PDF) as {PAPER_FILE}, upload it to Colab and run again.")


def get_supplements(urls, outdir="supp", timeout=300):
    """urls: {label: url}. Download each file once; return the local paths."""
    os.makedirs(outdir, exist_ok=True)
    paths = []
    for label, url in urls.items():
        path = os.path.join(outdir, url.rsplit("/", 1)[-1])
        if not os.path.exists(path):
            r = requests.get(url, timeout=timeout)
            r.raise_for_status()
            if "text/html" in r.headers.get("content-type", ""):
                raise ValueError(f"{label}: {url} returned a web page, not a data file")
            with open(path, "wb") as f:
                f.write(r.content)
        paths.append(path)
    return paths

def clean_table(raw, min_filled=0.5):
    """raw = a sheet read with header=None. Find the header row, name the columns, convert numbers."""
    raw = raw.dropna(how="all").dropna(axis=1, how="all").reset_index(drop=True)
    width = raw.shape[1]
    header_at = None
    for i in range(min(15, len(raw))):
        if raw.iloc[i].notna().sum() >= min_filled * width:
            header_at = i
            break
    if header_at is None:
        raise ValueError("could not find a header row in the first 15 rows")
    names, seen = [], {}
    for c in raw.iloc[header_at]:
        c = str(c).strip()
        seen[c] = seen.get(c, 0) + 1
        names.append(c if seen[c] == 1 else f"{c}.{seen[c]}")
    df = raw.iloc[header_at + 1:].copy()
    df.columns = names
    df = df.reset_index(drop=True)
    for c in df.columns:
        conv = pd.to_numeric(df[c], errors="coerce")
        filled = df[c].notna().sum()
        if filled and conv.notna().sum() >= 0.8 * filled:
            df[c] = conv
    return df          # missing values are KEPT: in proteomics a missing value means "not detected"


def sample_to_cell_line(name):
    """AU565_BREAST_TenPx05 -> AU565_BREAST (drops the TMT-plex suffix)."""
    return re.sub(r"_TenPx\d+$", "", str(name))

def protein_long(S2, gene_col):
    """One row per (cell_line, gene) with the protein level. Replicate samples are averaged."""
    sample_cols = [c for c, t in zip(S2.columns, S2.dtypes) if t.kind in "fi" and c != gene_col]
    tenpx = [c for c in sample_cols if re.search(r"_TenPx\d+$", str(c))]
    if len(tenpx) >= 100:             # annotation columns (peptide counts, ...) are numeric too: keep only the sample columns
        sample_cols = tenpx
    long = S2[[gene_col] + sample_cols].melt(id_vars=gene_col, var_name="sample", value_name="protein")
    long["cell_line"] = long["sample"].map(sample_to_cell_line)
    long = long.rename(columns={gene_col: "gene"}).dropna(subset=["protein"])
    return long.groupby(["cell_line", "gene"], as_index=False)["protein"].mean()

def build_dataset(S2, gene_col, panel_genes, panel_file="ccle_panel.csv"):
    prot = protein_long(S2, gene_col)
    prot = prot[prot["gene"].isin(panel_genes)]
    if os.path.exists(panel_file):
        om = pd.read_csv(panel_file)
        data = prot.merge(om, on=["cell_line", "gene"], how="inner")
    else:
        print(f"NOTE: {panel_file} not found, so the table has protein levels only (no RNA, copy number, mutation).")
        data = prot
    return data.sort_values(["gene", "cell_line"]).reset_index(drop=True)

_safety = dict(get_paper=get_paper, get_supplements=get_supplements, clean_table=clean_table)
_safety_build = build_dataset
print('safety net ready')

## Part 1. Get the paper (20 min)

**The task.** Write a function that takes a PubMed Central id and returns the paper's **title and full text** (and the sections, when the source has them). We use **Europe PMC**, a free service. It can serve the full text of some papers as XML:

`https://www.ebi.ac.uk/europepmc/webservices/rest/<PMCID>/fullTextXML`

XML is text with tags (`<sec>`, `<title>`, `<p>`). Taking the text out of tags without losing half of it is the first trap.

**The second trap is real: not every paper is served that way.** The paper behind our data (Nusinow 2020) is an author manuscript, and Europe PMC answered HTTP 500 for it. For the open-access paper we read here the XML should work, but a good function should not assume it. A good function **tries the routes in order and says what failed**: first the XML, then the PDF of the paper (`https://europepmc.org/api/getPdf?pmcid=<PMCID>`, text taken out with `pypdf`), and as a last resort a file `paper.txt` that you saved by hand (copy the text from the PubMed Central page or the PDF and upload it to Colab).

### 1a. A weak prompt and a stronger one

> **Weak:** "Write python to download the Jarnuczak cancer proteomics paper."
>
> **Stronger:** "Write a Python function `get_paper(pmcid)` using `requests`, `xml.etree.ElementTree` and `pypdf`. Try, in this order: (1) the Europe PMC endpoint `https://www.ebi.ac.uk/europepmc/webservices/rest/{pmcid}/fullTextXML`; (2) the PDF at `https://europepmc.org/api/getPdf?pmcid={pmcid}`; (3) a local file `paper.txt` if it exists. Return a dict with `title` (str), `sections` (a list of `(heading, text)` tuples; one `('full text', text)` item is fine for a PDF), `text` (the whole paper as one string, whitespace collapsed) and `source` (which route worked). Do not truncate anything. Raise a `ValueError` with the id in the message if the id is not `PMC` followed by digits. If every route fails, raise a `RuntimeError` that says what each route answered. Use a timeout."

Why is the second better? The AI cannot guess the **source**, the **keys**, whether to **keep everything**, or the **error behaviour**.

**Your turn.** Write your own prompt.

In [ ]:
prompt1 = """
WRITE YOUR PROMPT HERE
"""
REQ1 = {**COMMON, "Europe PMC source": r"europe ?pmc|ebi\.ac\.uk|pubmed central|\bpmc\b", "do not truncate": r"truncat|whole|entire|all of|full text|every"}
score1 = grade_prompt(prompt1, REQ1)

### 1b. Paste the AI's code

Copy the code your assistant gives you into the cell, replacing the placeholder. Keep the name `get_paper(pmcid)`.

In [ ]:
# PASTE THE AI'S CODE HERE (keep the name get_paper)
def get_paper(pmcid):
    raise NotImplementedError("paste your AI-written function here")

### 1c. Check it against what you know

Three checks that need no knowledge of the code:

1. **Size.** A research paper is tens of thousands of characters. A short result means something was cut.
2. **Numbers you can find.** The paper's abstract says **191** cancer cell lines and a median protein-to-RNA correlation of **0.58**, and the results say **7,171** MS runs were reanalysed. If those are missing, text is missing.
3. **A bad id.** The function must complain clearly, not return an empty result.

In [ ]:
def verify_part1(fn, pmcid=PMCID):
    ok = True
    try:
        p = fn(pmcid)
    except Exception as e:
        print("FAIL: your function raised for a valid id:", type(e).__name__, str(e)[:200]); return False
    for k in ("title", "sections", "text"):
        if k not in p:
            print(f"FAIL: missing key {k!r}"); ok = False
    if not ok: return False
    n = len(p["text"])
    print(f"title   : {p['title'][:90]}")
    print(f"sections: {len(p['sections'])}   characters: {n:,}")
    if len(p["title"]) < 10: print("FAIL: no sensible title"); ok = False
    print(f"source  : {p.get('source', '(not reported)')}")
    if len(p["sections"]) < 1: print("FAIL: no sections returned"); ok = False
    if n < 20000: print("FAIL: the text is very short for a full paper (under 20,000 characters). Was it cut?"); ok = False
    for needle in ("191", "0.58", "7,171"):
        if needle not in p["text"]: print(f"FAIL: {needle!r} is in the paper but not in your text"); ok = False
    try:
        fn("not-an-id")
        print("FAIL: an invalid id should raise an error, but nothing was raised"); ok = False
    except Exception as e:
        print("bad id  : raised", type(e).__name__, "->", str(e)[:100])
        if not re.search(r"not-an-id|PMC|id", str(e), flags=re.I): print("note: the message does not mention the id");
    print("PASS" if ok else "NOT YET: fix the function (or your prompt) and run again")
    return ok

ok1 = verify_part1(get_paper)

### 1d. Hunt the bug

Below is an AI answer that is plausible and **runs**. Run the check on it, then find out why it fails.

In [ ]:
def get_paper_flawed(pmcid):
    r = requests.get(f"https://www.ebi.ac.uk/europepmc/webservices/rest/{pmcid}/fullTextXML", timeout=60)
    import xml.etree.ElementTree as ET
    root = ET.fromstring(r.content)
    title = "".join(root.find(".//article-title").itertext()).strip()
    text = " ".join("".join(root.itertext()).split())[:5000]        # "keep it manageable"
    secs = [("", text)]
    return {"title": title, "sections": secs, "text": text}

_ = verify_part1(get_paper_flawed)

**Questions.** (1) Which line made the text short, and why is that dangerous when you then ask "how many cell lines were profiled?" (2) The function also never fails on a bad id. What would happen later in your pipeline?

### 1e. Fix your prompt

Add the sentence that would have prevented the bug and grade it again.

In [ ]:
prompt1_fixed = """
COPY YOUR PROMPT AND ADD THE SENTENCES THAT WOULD HAVE PREVENTED THE BUG
"""
_ = grade_prompt(prompt1_fixed, REQ1)

## Part 2. Supplementary tables (25 min)

The proteins, the cell-line list and the metadata are in the **supplementary tables of Nusinow et al. 2020**: Table S1 (the cell lines) and Table S2 (the protein levels, one row per protein and one column per sample). Europe PMC does not serve them for that paper (its supplementary-files route returned no zip in the first Colab run), but the authors' lab publishes the same tables for download. This is common: **the paper points to the data, and the data live somewhere else.**

You will write two functions: `get_supplements(urls)` that downloads the files and returns their paths, and `clean_table(raw)` that turns a messy sheet into a proper data frame.

For Table S2 we use the lab's **compressed CSV version** of the normalized table (the same data as the Excel file, much faster to read). The lab updates its files, so the "current" table can differ slightly from the one printed with the paper in 2020.

**What makes supplementary tables messy.** Title rows above the header, notes under the table, several sheets per file, numbers stored as text, and **missing values that mean something**. In proteomics, an empty cell means *not detected in that sample*, which is information, not an error.

### 2a. Prompts

In [ ]:
SUPP_URLS = {
    "Table S1 (sample information)": "https://gygi.hms.harvard.edu/data/ccle/Table_S1_Sample_Information.xlsx",
    "Table S2 (normalized protein quantification)": "https://gygi.hms.harvard.edu/data/ccle/protein_quant_current_normalized.csv.gz",
}

In [ ]:
prompt2 = """
WRITE YOUR PROMPT HERE. Ask for TWO functions:
  get_supplements(urls) -> list of file paths       (urls is a dict {label: url}; download each file)
  clean_table(raw) -> DataFrame                      (raw is a sheet read with header=None)
Say what the messy parts are, and what to do with missing values.
"""
REQ2 = {**COMMON, "two functions": r"get_supplements.*clean_table|clean_table.*get_supplements",
        "header row": r"header|title row|column names",
        "missing values": r"missing|nan|empty|not detected|dropna|keep"}
score2 = grade_prompt(prompt2, REQ2)

### 2b. Paste the AI's code

In [ ]:
# PASTE THE AI'S CODE HERE (keep the names get_supplements and clean_table)
def get_supplements(urls):
    raise NotImplementedError("paste your AI-written function here")

def clean_table(raw):
    raise NotImplementedError("paste your AI-written function here")

### 2c. Download, look, and choose the tables

First look at what you got. Reading a big Excel file takes a minute or two. This cell downloads, reads **every sheet**, and prints a catalogue. Then it guesses which table is the cell-line list (S1) and which is the protein matrix (S2): the protein matrix is the **widest** table. Check the guess by eye and override it if needed.

In [ ]:
try:
    paths = get_supplements(SUPP_URLS)
    print("your get_supplements worked")
except Exception as e:
    print("your get_supplements failed:", type(e).__name__, str(e)[:120], "-> using the safety net")
    paths = _safety["get_supplements"](SUPP_URLS)

frames = {}
for p in paths:
    if p.lower().endswith((".xlsx", ".xls")):
        for sheet, df in pd.read_excel(p, sheet_name=None, header=None).items():
            frames[f"{os.path.basename(p)} :: {sheet}"] = df
    elif p.lower().endswith((".csv", ".csv.gz", ".tsv", ".tsv.gz", ".txt")):
        sep = "\t" if ".tsv" in p.lower() or p.lower().endswith(".txt") else ","
        frames[os.path.basename(p)] = pd.read_csv(p, sep=sep, header=None, low_memory=False)

print(f"\n{len(frames)} tables found\n")
for i, (k, df) in enumerate(frames.items()):
    first = [str(x)[:25] for x in df.iloc[0].dropna().tolist()[:3]]
    print(f"[{i}] {k[:60]:60s} rows={df.shape[0]:>6}  cols={df.shape[1]:>4}  first cells: {first}")

S2_KEY = max(frames, key=lambda k: frames[k].shape[1])                       # widest = protein matrix
S1_KEY = max([k for k in frames if k != S2_KEY and frames[k].shape[1] <= 80], key=lambda k: frames[k].shape[0], default=None)
print("\nguess  S1 (cell lines):", S1_KEY)
print("guess  S2 (proteins)  :", S2_KEY)
# If the guess is wrong, set the keys by hand, for example:  S1_KEY = list(frames)[0]

### 2d. Check your `clean_table`

You do not need to know the code to check it. The checks use things you know about the Nusinow paper:

- **S1** should list about **375** cell lines (a few more if replicates are listed).
- **S2** should have **thousands** of proteins and **hundreds** of sample columns, and should still contain proteins that have some missing values.
- **ERBB2** (HER2) is a known protein. The paper shows it is very high in **breast** cancer lines (Figure 1D-E). Its top samples should be breast lines.

In [ ]:
def find_gene_col(df):
    for c in df.columns:
        if re.search(r"gene", str(c), flags=re.I): return c
    for c, t in zip(df.columns, df.dtypes):
        if t == object: return c

def verify_part2(fn, raw1, raw2):
    ok = True
    try:
        S1, S2 = fn(raw1), fn(raw2)
    except Exception as e:
        print("FAIL: your clean_table raised:", type(e).__name__, str(e)[:200]); return False, None, None
    print(f"S1: {S1.shape[0]} rows x {S1.shape[1]} cols    S2: {S2.shape[0]} rows x {S2.shape[1]} cols")
    if abs(len(S1) - 375) > 30:
        print(f"FAIL: S1 has {len(S1)} rows; the paper profiled 375 cell lines (replicates may add a few)"); ok = False
    num = [c for c, t in zip(S2.columns, S2.dtypes) if t.kind in "fi"]
    samples = [c for c in num if "_TenPx" in str(c)]        # real sample columns look like AU565_BREAST_TenPx05
    if len(samples) >= 100:
        num = samples                                         # drop Group_ID, peptide counts and other numeric extras
    if S2.shape[0] < 5000:
        print(f"FAIL: S2 has only {S2.shape[0]} proteins; the paper quantified thousands. Were rows with any missing value dropped?"); ok = False
    if len(num) < 300:
        print(f"FAIL: only {len(num)} numeric sample columns; expected 300 or more. Are numbers stored as text?"); ok = False
    if len(num) and S2[num].isna().sum().sum() == 0:
        print("FAIL: no missing values at all. In proteomics some proteins are not detected in some samples; they should be kept as NaN."); ok = False
    g = find_gene_col(S2)
    if g is None or isinstance(S2[g], pd.DataFrame):
        hit = S2.iloc[0:0]            # no usable gene column (or duplicated column names)
    else:
        hit = S2[S2[g].astype(str).str.upper() == "ERBB2"]
    if len(hit) == 0:
        print(f"FAIL: ERBB2 not found in column {g!r}"); ok = False
    else:
        top = hit.iloc[0][num].astype(float).sort_values(ascending=False).head(3)
        print("ERBB2 top samples:", ", ".join(top.index.astype(str)))
        if not any("BREAST" in str(s).upper() for s in top.index):
            print("WARNING: none of the top ERBB2 samples is a breast line. Check the column names and the orientation.")
    print("PASS" if ok else "NOT YET")
    return ok, S1, S2

def clean_with_fallback(raw):
    try: return clean_table(raw)
    except Exception: return _safety["clean_table"](raw)

ok2, S1, S2 = verify_part2(clean_table, frames[S1_KEY], frames[S2_KEY])

### 2e. Hunt the bug

This AI answer is short and **runs without any error**. Run the check on it.

In [ ]:
def clean_table_flawed(raw):
    df = raw.copy()
    df.columns = df.iloc[0]
    df = df.iloc[1:].dropna()            # "remove incomplete rows"
    return df.reset_index(drop=True)

_ = verify_part2(clean_table_flawed, frames[S1_KEY], frames[S2_KEY])

**Questions.** (1) How many proteins did `dropna()` remove, and why does a protein with *one* missing sample get thrown away? (2) What would a downstream model learn if the proteins that are not detected in many lines were silently removed? (3) Which sentence in the prompt would have stopped it?

### 2f. Fix your prompt

In [ ]:
prompt2_fixed = """
COPY YOUR PROMPT AND ADD THE SENTENCES THAT WOULD HAVE PREVENTED THE BUG
"""
_ = grade_prompt(prompt2_fixed, REQ2)

# Make sure S1 and S2 exist for the next parts, even if your function failed:
if S2 is None:
    S1, S2 = clean_with_fallback(frames[S1_KEY]), clean_with_fallback(frames[S2_KEY])
    print("using safety-net tables:", S1.shape, S2.shape)

## Part 3. Facts from the text, with evidence (20 min)

Tables are the easy half. The **text** holds facts that are in no table: how many cell lines, what the correlation was, how replicates behaved. An LLM can read a section and answer in a structured form. The danger is that it can also **invent a number, or invent the sentence that supports it**.

The rule for this part: **every extracted value must come with a quoted sentence, and the quote must be a verbatim sentence from the paper.** (The paper of Part 1: Jarnuczak et al. 2021.) A quote that is not in the paper is the signature of a hallucination, and you can check it with one line of code.

### 3a. Ask your assistant

Open your AI assistant (any chat). Paste the **abstract and the Results section** of the paper (run the cell below to print and copy them) and ask for the four facts below as JSON. Then paste the JSON into the next cell.

The facts to extract:

| fact | what to look for |
| --- | --- |
| `number of cancer cell lines` | a count |
| `number of MS runs reanalysed` | a count (with a thousands separator) |
| `number of lineages` | a count |
| `median protein-RNA correlation across cell lines` | a correlation coefficient |

In [ ]:
try:
    PAPER = get_paper(PMCID)
except Exception:
    PAPER = _safety["get_paper"](PMCID)
results = [t for h, t in PAPER["sections"] if re.search(r"result", h, flags=re.I)]
RESULTS_TEXT = (PAPER.get("abstract", "") + " " + results[0]).strip() if results else PAPER["text"]
print(f"Abstract and results: {len(RESULTS_TEXT):,} characters. Select the text below and copy it into your assistant.\n")
print(RESULTS_TEXT[:1500] + " ...")

In [ ]:
prompt3 = """
WRITE YOUR PROMPT HERE. For each fact ask for: fact, value, evidence (an exact sentence copied from the text).
Say what to do if the text does not contain the fact.
"""
REQ3 = {"JSON output": r"json",
        "evidence quote": r"quote|verbatim|exact sentence|evidence|copied",
        "no guessing": r"not (stated|found|in)|null|unknown|do not guess|don't guess|do not invent|only (from|use)|if the text",
        "the four facts": r"cell lines.*lineage|lineage.*cell lines",
        "format example": r"\{|\[|example"}
score3 = grade_prompt(prompt3, REQ3)

### 3b. Paste the AI's answer and verify it

Paste the JSON your assistant gave you between the triple quotes. No assistant? Leave it as it is: the notebook then uses a pre-made answer so you can still run the check.

In [ ]:
AI_JSON = """
PASTE THE JSON FROM YOUR ASSISTANT HERE
"""

In [ ]:
def sentences(text):
    return re.split(r"(?<=[.!?])\s+(?=[A-Z(])", text)

def find_sentence(text, pattern):
    for s in sentences(text):
        if re.search(pattern, s):
            return s.strip()
    return ""


EXPECTED = {"number of cancer cell lines": "191", "number of MS runs reanalysed": "7,171", "number of lineages": "13",
            "median protein-RNA correlation across cell lines": "0.58"}

def norm(s):
    s = str(s).lower().replace("−", "-").replace("–", "-").replace("’", "'")
    return " ".join(s.split())

def verify_facts(facts, paper_text):
    """For each fact: is the quote in the paper, is the value in the quote, is the value what the paper says?"""
    full = norm(paper_text)
    rows = []
    for f in facts:
        ev, val = norm(f.get("evidence", "")), norm(f.get("value", ""))
        in_paper = len(ev) > 15 and ev in full
        in_quote = bool(val) and val in ev
        exp = EXPECTED.get(f.get("fact"))
        rows.append({"fact": f.get("fact"), "value": f.get("value"),
                     "quote is verbatim in paper": in_paper, "value is in the quote": in_quote,
                     "value matches paper": (norm(exp) == val) if exp else None})
    out = pd.DataFrame(rows)
    return out

if "PASTE THE JSON" in AI_JSON:
    print("No answer pasted: using the pre-made flawed answer below so you can see the check work.")
    AI_JSON = json.dumps([
        {"fact": "number of cancer cell lines", "value": "191", "evidence": find_sentence(PAPER["text"], r"\b191 cancer cell lines")},
        {"fact": "number of MS runs reanalysed", "value": "7,171", "evidence": "The authors collected and re-annotated more than seven thousand raw files from eleven studies."},
        {"fact": "number of lineages", "value": "13", "evidence": "The reference map covers thirteen distinct tissue lineages."},
        {"fact": "median protein-RNA correlation across cell lines", "value": "0.6", "evidence": find_sentence(PAPER["text"], r"\b191 cancer cell lines")}])
facts = json.loads(AI_JSON)
report = verify_facts(facts, PAPER["text"])
display(report)
bad = report[(~report["quote is verbatim in paper"]) | (~report["value is in the quote"]) | (report["value matches paper"] == False)]
print("ALL FACTS VERIFIED" if bad.empty else f"{len(bad)} of {len(report)} facts FAILED the check: {', '.join(bad['fact'])}")

### 3c. What went wrong

If you used the pre-made answer, each failing fact shows a different kind of failure:

- a **paraphrased** quote (the sentence is not in the paper even though the idea is),
- a **rounded** value with an invented supporting sentence (the model wrote the evidence to fit its answer),
- a **real sentence from the wrong place** (it contains no `60`).

All of these *look* reasonable to a human reader. Only the verbatim test caught them. Edit your prompt below so that the assistant is told to copy sentences exactly and to answer `null` when the text does not say. Then ask your assistant again, paste the new JSON, and re-run the check.

In [ ]:
prompt3_fixed = """
COPY YOUR PROMPT AND ADD THE SENTENCES THAT WOULD HAVE PREVENTED THE FAILURES
"""
_ = grade_prompt(prompt3_fixed, REQ3)

## Part 4. Build the shared dataset (13 min)

Everything you retrieved so far becomes **one table** that the next notebooks use.

**The table.** One row per *(cell line, gene)*. Columns:

| Column | Layer | Source |
| --- | --- | --- |
| `cell_line` | key | CCLE name, for example `AU565_BREAST` |
| `gene` | key | gene symbol |
| `protein` | proteomic | Table S2 of the paper (this notebook) |
| `rna_log2tpm` | transcriptomic | class file `ccle_panel.csv` |
| `copy_number` | genomic | class file `ccle_panel.csv` |
| `mutation` | genomic | class file `ccle_panel.csv`, protein change such as `p.R175H`, blank if none |

We keep a **panel** of cancer genes, not all 12,000 proteins, so the file is small and every row is interpretable.

Two details that cause silent errors:
- **Sample names.** Table S2 column names look like `AU565_BREAST_TenPx05`: cell line plus the mass-spec batch. The suffix must be dropped to match the other layers, and cell lines measured in several batches are averaged.
- **Missing values.** A protein not detected in a line stays missing. Do not fill it with zero.

### 4a. Ask your assistant

In [ ]:
# 20 cancer genes. In the Nusinow data ESR1 has no protein, and PGR (about 36 cell lines) and PALB2 (about 160) have little,
# so the finished table has 19 genes, and those two genes cover far fewer cell lines than the rest.
PANEL = ["BRCA1", "BRCA2", "TP53", "ERBB2", "ESR1", "PGR", "PIK3CA", "PTEN", "EGFR", "MYC",
         "ATM", "CHEK2", "PALB2", "RAD51", "MLH1", "MSH2", "KRAS", "CDKN2A", "RB1", "AKT1"]

prompt4 = """
WRITE YOUR PROMPT HERE. Ask for build_dataset(S2, gene_col, panel_genes, panel_file) -> DataFrame
Describe the layout of S2, the sample-name suffix, the merge keys and the missing-value rule.
"""
REQ4 = {**COMMON, "sample names": r"suffix|_TenPx|strip|sample name|replicate|average|mean",
        "merge keys": r"merge|join|key|cell_line|on=",
        "melt / long format": r"melt|long|tidy|one row per"}
score4 = grade_prompt(prompt4, REQ4)

In [ ]:
# PASTE THE AI'S CODE HERE (keep the name build_dataset)
def build_dataset(S2, gene_col, panel_genes, panel_file="ccle_panel.csv"):
    raise NotImplementedError("paste your AI-written function here")

### 4b. Check the dataset, using the paper itself

Beyond counts, there is a **result the papers report** that your table should reproduce: protein and RNA levels of the same gene across cell lines are positively correlated (Nusinow et al. 2020 report a mean of about **0.48** over all genes; Jarnuczak et al. 2021 report a median of **0.58** across cell lines, Part 3). If your merge shifted rows, matched the wrong cell lines or mixed up genes, this correlation collapses toward zero. That is the strongest check you have, because it tests the *biology*, not the code.

In [ ]:
def verify_part4(data):
    ok = True
    print(f"rows: {len(data):,}   genes: {data['gene'].nunique()}   cell lines: {data['cell_line'].nunique()}   columns: {list(data.columns)}")
    if data.duplicated(["cell_line", "gene"]).any():
        print("FAIL: duplicate (cell_line, gene) rows; replicate samples were not averaged"); ok = False
    if data["cell_line"].str.contains("TenPx", regex=False).any():
        print("FAIL: batch suffix (_TenPx) still in the cell line names"); ok = False
    if (data["protein"] == 0).mean() > 0.2:
        print("FAIL: many protein values are exactly 0; were missing values filled with zero?"); ok = False
    if data["gene"].nunique() < 10:
        print("FAIL: fewer than 10 panel genes survived"); ok = False
    if "rna_log2tpm" in data.columns:
        per = []
        for g, d in data.groupby("gene"):
            d = d.dropna(subset=["protein", "rna_log2tpm"])
            if len(d) >= 30: per.append(spearmanr(d["protein"], d["rna_log2tpm"])[0])
        m = float(np.nanmean(per)) if per else float("nan")
        print(f"mean protein-vs-RNA Spearman over panel genes: {m:.2f}   (Nusinow et al. report about 0.48 over all genes)")
        if not m > 0.2:
            print("FAIL: protein and RNA should correlate positively. Check that cell-line names matched across layers."); ok = False
    else:
        print("NOTE: no RNA column (class file not loaded), so the biology check was skipped.")
    print("PASS" if ok else "NOT YET")
    return ok

GENE_COL = find_gene_col(S2)
try:
    DATA = build_dataset(S2, GENE_COL, PANEL)
except Exception as e:
    print("your build_dataset failed:", type(e).__name__, str(e)[:150], "-> using the safety net")
    DATA = None
if DATA is None:
    DATA = _safety_build(S2, GENE_COL, PANEL)
ok4 = verify_part4(DATA) if DATA is not None else False

### 4c. Hunt the bug

This AI answer **runs** and gives a table of the right shape. Find out what is wrong.

In [ ]:
def build_dataset_flawed(S2, gene_col, panel_genes, panel_file="ccle_panel.csv"):
    sample_cols = [c for c, t in zip(S2.columns, S2.dtypes) if t.kind == "f"]
    long = S2[[gene_col] + sample_cols].melt(id_vars=gene_col, var_name="cell_line", value_name="protein")
    long = long.rename(columns={gene_col: "gene"})
    long["protein"] = long["protein"].fillna(0)                     # "avoid NaN problems"
    long = long[long["gene"].isin(panel_genes)]
    if os.path.exists(panel_file):
        long = long.merge(pd.read_csv(panel_file), on=["cell_line", "gene"], how="left")
    return long

_ = verify_part4(build_dataset_flawed(S2, GENE_COL, PANEL))

**Questions.** (1) Which bugs does the check name? (2) Look at `cell_line` in the flawed table: why does the merge with the RNA file silently find nothing, and why is `how="left"` hiding it? (3) A model trained on this table would still run. What would it learn?

### 4d. Save the shared file

In [ ]:
if DATA is not None:
    DATA.to_csv("ccle_multiomics_panel.csv", index=False)
    print("saved ccle_multiomics_panel.csv:", DATA.shape)
    display(DATA.head(8))
    # Optional: download it so you can reuse it in the next notebooks
    try:
        from google.colab import files
        files.download("ccle_multiomics_panel.csv")
    except Exception:
        pass

## Part 5. Wrap-up (2 min)

What you now have, all retrieved by code from a published paper and checked:

- the **full text** (facts, with verified quotes),
- the **proteome** of 375 cancer cell lines (Nusinow's Table S2),
- a shared **multi-omics panel table** (`ccle_multiomics_panel.csv`): protein, RNA, copy number and mutations for the same cell lines and genes.

### The checking habits

- Test against **something you know** (a count, a protein, a number in the text).
- Test the **edge case** (a bad id, a missing value, a different layout).
- Make claims **carry evidence** that code can verify (the verbatim quote).
- Check a **biological expectation** the paper itself reports (protein and RNA correlate).
- Never accept code because it ran.

| Bug you met | Part | A sentence that would have prevented it |
| --- | --- | --- |
| text silently cut at 5,000 characters | 1 | "do not truncate; return the full text and its length" |
| `dropna()` removing most proteins | 2 | "keep missing values as NaN; report how many rows and columns remain" |
| invented or paraphrased evidence | 3 | "copy each evidence sentence exactly; answer null if the text does not say" |
| batch suffix in names, zero-filled missing values, left merge hiding a failed match | 4 | "strip the suffix, average replicates, never fill missing with 0, use an inner merge and print how many rows matched" |

## Troubleshooting

| Symptom | Likely cause | Fix |
| --- | --- | --- |
| `NotImplementedError: paste your AI-written function here` | the placeholder is still there | Paste the AI's code over it and run the cell |
| `FAIL: your function raised ...` | the AI used a different function name or arguments | Rename to the expected name, or fix the prompt |
| `Could not get the text of PMC...` | Europe PMC serves neither the XML nor the PDF | Save the paper's text as `paper.txt` (copy from the PubMed Central page or the PDF), upload it, re-run |
| A download returns a web page, not a file | the lab site changed its links | Check `SUPP_URLS` in a browser; ask the instructor |
| Reading the Excel file is slow | Table S2 is large | Wait 1 to 3 minutes; it runs once |
| The guessed S1 or S2 table is wrong | the catalogue picks by shape | Set `S1_KEY` and `S2_KEY` by hand from the catalogue |
| No `ccle_panel.csv` | the class file was not uploaded | Part 4 still runs with protein levels only |